In [2]:
import pandas as pd
import numpy as np
import random

In [194]:

def get_entropy(y: pd.Series):
    unique_labels = y.unique()
    
    if len(unique_labels) == 1:
        return 0
    
    n = len(y)
    count_1 = y.sum()
    count_0 = n - count_1

    proba_0 = count_0 / n
    proba_1 = count_1 / n

    return - (np.log2(proba_0) * proba_0 + np.log2(proba_1) * proba_1)

def get_gini(y: pd.Series):
    unique_labels = y.unique()

    if len (unique_labels) == 1:
        return 0

    n = len(y)
    count_1 = y.sum()
    count_0 = n - count_1

    proba_0 = count_0 / n
    proba_1 = count_1 / n

    return 1 - proba_0 ** 2 - proba_1 ** 2
 
class Leaf:
    def __init__(self, parent, samples, proba, depth):
        self.parent = parent
        self.samples = samples
        self.proba = proba
        self.depth = depth

    def __str__(self):
        return f'{"  " * self.depth} {self.proba}'

class Node:
    def __init__(self, parent=None):
        self.parent = parent
        self.left = None
        self.right = None
        self.samples = None
        self.depth = 1
        self.split_value = None
        self.column = None
        self.ig = None
        self.is_left = None

    def __str__(self):
        return  '\n'.join([
        f'{"  " * self.depth} {self.column} {self.split_value}',
        f'{self.left}',
        f'{self.right}'])

    def predict_proba(self, row: pd.Series):
        if row.loc[self.column] <= self.split_value:
            if isinstance(self.left, Leaf):
                return self.left.proba
            else: 
                return self.left.predict_proba(row)
        else:
            if isinstance(self.right, Leaf):
                return self.right.proba
            else: 
                return self.right.predict_proba(row)

class MyTreeClf:
    def __init__(self, max_depth: int = 5, min_samples_split: int = 2,
                 max_leafs: int = 20, bins=None, criterion="entropy"):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_leafs = max_leafs
        self.nodes_cnt = 1
        self.leafs_cnt = None
        self.bins = bins
        self.split_values = None
        self.criterion = criterion
        self.fi = dict()
        self.N = None

    def __str__(self):
        return f"MyTreeClf class: max_depth={self.max_depth}, min_samples_split={self.min_samples_split}, max_leafs={self.max_leafs}"

    def __build_split_values(self, X):
        self.split_values = dict()
        for column in X.columns:
            unique_values = X[column].unique()
            if (len(unique_values) - 1 <= self.bins - 1):
                sorted_unique_values = np.sort(unique_values)
                self.split_values[column] = (sorted_unique_values[:-1] + sorted_unique_values[1:]) / 2
            else:
                count_elements, split_values = np.histogram(X[column], bins = self.bins)
                self.split_values[column] = split_values[1:-1]

    def get_split_values(self, X, column):
        if self.bins is None:
            unique_values = X[column].unique()
            sorted_unique_values = np.sort(unique_values)
            return (sorted_unique_values[:-1] + sorted_unique_values[1:]) / 2
        else:
            max_value = X[column].max()
            min_value = X[column].min()
            return self.split_values[column][ (min_value <= self.split_values[column]) & (max_value > self.split_values[column])]

    def __init_fi(self, X):
        for column in X.columns:
            self.fi[column] = 0 
    
    def fit(self, X: pd.DataFrame, y: pd.Series):
        self.root = Node()
        self.__init_fi(X)
        self.N = len(y)
        if self.bins is not None:
            self.__build_split_values(X)
            
        self.build_tree_recursive(self.root, X, y)
        self.leafs_cnt = self.nodes_cnt + 1

    def get_best_split(self, X: pd. DataFrame, y : pd.Series):

        igs = []
        
        for column in X.columns:
            split_values = self.get_split_values(X, column)
            if self.criterion == "entropy":
                criterion_function = get_entropy
            elif self.criterion == "gini":
                criterion_function = get_gini
            criterion = criterion_function(y)
            
            for split_value in split_values: 

                left_indexes = X.index[X[column] <= split_value]
                left_y = y.loc[left_indexes]
                left_criterion = criterion_function(left_y)
                
                right_indexes = X.index[X[column] > split_value]
                right_y = y.loc[right_indexes]
                right_criterion = criterion_function(right_y)
    
                ig = criterion - left_criterion * len(left_indexes) / len(y) - right_criterion * len(right_indexes) / len(y) 
    
                igs.append((column, split_value, ig))
        if igs:    
            return max(igs, key= lambda x: x[2])
        else:
            return None, None, None

    
    def predict_proba(self, X: pd.DataFrame) -> pd.Series:
        probas = []
        for index, row in X.iterrows():
            proba = self.root.predict_proba(row)
            probas.append(proba)
        return pd.Series(probas, X.index)

    def predict(self, X: pd.DataFrame) -> pd.Series:
        probas = self.predict_proba(X)
        return probas > 0.5
        
    def build_tree_recursive(self, node: Node, X, y):

        node.column, node.split_value, node.ig = self.get_best_split(X, y)
        
        if node.column is not None:
            self.fi[node.column] += len(y) / self.N * node.ig
            
            left_indexes = X.index[X[node.column] <= node.split_value]
            left_cnt = len(left_indexes)
            left_y = y.loc[left_indexes]
            
            if len(left_y.unique()) == 1 or left_cnt < self.min_samples_split or \
                self.nodes_cnt + 1 >= self.max_leafs or node.depth == self.max_depth \
                :
                node.left = Leaf(node, len(left_y), left_y.mean(), node.depth + 1)
            else:
                node.left = Node(node)
                node.left.depth = node.depth + 1
                node.left.is_left = True
                self.nodes_cnt += 1
                self.build_tree_recursive(node.left, X.loc[left_indexes], left_y)
    
            right_indexes = X.index[X[node.column] > node.split_value]
            right_cnt = len(right_indexes)
            right_y = y.loc[right_indexes]
            
            if len(right_y.unique()) == 1  or right_cnt < self.min_samples_split  or \
                self.nodes_cnt + 1 >= self.max_leafs or node.depth == self.max_depth:
                node.right = Leaf(node, len(right_y), right_y.mean(), node.depth + 1)
            else:
                node.right = Node(node)
                node.right.depth = node.depth + 1
                node.right.is_left = False
                self.nodes_cnt += 1
                self.build_tree_recursive(node.right, X.loc[right_indexes], right_y)
        else:
            parent = node.parent
            if node.is_left:
                parent.left = Leaf(parent, len(y), y.mean(), parent.depth + 1)
            else:
                parent.right = Leaf(parent, len(y), y.mean(), parent.depth + 1)

In [195]:

df = pd.read_csv('./data_banknote_authentication.txt', header=None)
df.columns = ['variance', 'skewness', 'curtosis', 'entropy', 'target']
X, y = df.iloc[:,:4], df['target']

In [199]:
my_tree_clf = MyTreeClf(5,200,10,4,"entropy")
my_tree_clf.fit(X, y)

In [200]:
print(my_tree_clf.fi)

{'variance': 0.42755468161889615, 'skewness': 0.17088402599288008, 'curtosis': 0.12838484105317446, 'entropy': 0}
